# 🍏 Apple Generative Imagery Systems - Real-Time Cloud Studio (NVIDIA GPU)
### SDXL (20-Step Quality / 4-Step Lightning) + ControlNet Depth + Custom `apple_minimal_craft` LoRA
---
Google Colab의 **NVIDIA GPU**를 사용하여 에러 없이 **장당 2~3초** 만에 4종의 Apple 스탠다드 룩디벨롭 에셋을 즉시 일괄 렌더링합니다.

In [ ]:
# 1. 필수 라이브러리 일괄 설치 (PEFT 포함으로 LoRA 에러 원천 차단)
!pip install -q diffusers transformers accelerate safetensors opencv-python peft huggingface_hub

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 2. [선택] 맥북에서 새로 수정한 3D Depth 맵 업로드
from google.colab import files
import shutil, os

print("📤 맥북의 depth_controlnet_1080p.png 파일이 필요하면 선택하세요 (없으면 Drive에서 자동 검색):")
try:
    uploaded = files.upload()
    for fn in uploaded.keys():
        shutil.copy(fn, "/content/depth_controlnet_1080p.png")
        print(f"✅ 새 3D Depth 맵 업로드 완료: {fn}")
except Exception as e:
    print("업로드 건너뜀 (기존 Drive 파일 사용)")

In [ ]:
# 3. 🚀 안정적인 고화질 AI 파이프라인 로드
import os, glob, torch
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, DPMSolverMultistepScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using Device: {device}")

# 1. 3D Depth 이미지 로드 및 리사이즈 (1024x576)
depth_img_path = "/content/depth_controlnet_1080p.png"
if not os.path.exists(depth_img_path):
    depth_files = glob.glob("/content/drive/MyDrive/**/depth_controlnet_1080p.png", recursive=True)
    depth_img_path = depth_files[0] if depth_files else None

if depth_img_path and os.path.exists(depth_img_path):
    print(f"✅ 3D Depth Map 연동 완료: {depth_img_path}")
    control_image = Image.open(depth_img_path).convert("RGB").resize((1024, 576))
else:
    print("⚠️ 3D Depth 이미지를 찾지 못해 기본 흰 캔버스를 사용합니다.")
    control_image = Image.new("RGB", (1024, 576), (0, 0, 0))

# 2. SDXL ControlNet & Base 모델 로드
print("📦 SDXL 및 ControlNet 로딩 중...")
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-depth-sdxl-1.0",
    torch_dtype=torch.float16
)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

# 3. 최고급 DPM++ 2M Karras 스케줄러 세팅
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

# 4. 우리가 훈련한 Apple Craft LoRA 연동
lora_files = glob.glob("/content/drive/MyDrive/**/apple_minimal_craft_sdxl_v1.safetensors", recursive=True)
if lora_files:
    pipe.load_lora_weights(lora_files[0])
    print(f"✅ Apple Craft LoRA 장착 완료: {lora_files[0]}")
else:
    print("⚠️ Drive에서 LoRA 파일을 찾지 못해 기본 SDXL 스타일로 진행합니다.")

print("✨ 모든 렌더링 파이프라인 준비 완료!")

In [ ]:
# 4. 🍏 4종 재질 룩디벨롭 일괄 고화질 렌더링 (장당 3~4초!)
prompts = {
    "01_Pastel_Peach_Cream": "a hero commercial product photo in apl_minimal_craft style of organic pastel peach cosmetic cream swatch with elegant fluid swirls, rich viscous creamy texture, delicate specular studio rim lighting, tactile surface, clean macro photography, on a clean light grey studio tabletop background",
    "02_Translucent_Hydrating_Gel": "a hero commercial product photo in apl_minimal_craft style of crystal clear translucent hydrating cosmetic gel swatch with micro bubbles, glossy wet reflection, delicate studio rim light, on a clean light grey studio background",
    "03_Terracotta_Mineral_Clay": "a hero commercial product photo in apl_minimal_craft style of textured terracotta mineral clay cosmetic paste swatch, rich earthy granular surface, soft daylight, warm minimalist aesthetic, on a light grey studio background",
    "04_Golden_Honey_Balm": "a hero commercial product photo in apl_minimal_craft style of luminous golden honey cosmetic balm swatch with micro exfoliating botanical particles, radiant glossy finish, elegant studio lighting, on a light grey studio background"
}

negative_prompt = "flat, plastic board, 3d polygon, rigid toy, flesh skin, blurry, deformed, noisy background, dark void, watermark, low quality, oversaturated"
output_dir = "/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/colab_hero_renders"
os.makedirs(output_dir, exist_ok=True)

import time
from IPython.display import display

print("🚀 고품질 상업용 룩디벨롭 렌더링 시작!\n")

for name, prompt in prompts.items():
    start_time = time.time()
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=control_image,
        num_inference_steps=20,
        guidance_scale=6.0,
        controlnet_conditioning_scale=0.45,
        control_guidance_end=0.65
    ).images[0]
    
    # 1080p 업스케일 저장
    final_image = image.resize((1920, 1080), Image.Resampling.LANCZOS)
    save_path = os.path.join(output_dir, f"{name}.png")
    final_image.save(save_path)
    
    elapsed = time.time() - start_time
    print(f"💎 [{name}] 생성 완료! (소요 시간: {elapsed:.2f}초) -> {save_path}")
    display(final_image.resize((640, 360)))

print("\n🎉 4종 모든 룩디벨롭 에셋 생성이 성공적으로 완료되었습니다!")